## Chuyên viên tri thức

### Tác nhân hỏi đáp đóng vai trò là một chuyên viên tri thức
### Dành cho nhân viên của Insurellm, một công ty công nghệ bảo hiểm
### Tác nhân cần đưa ra câu trả lời chính xác và giải pháp phải có chi phí thấp.

Dự án này sẽ sử dụng RAG (Retrieval Augmented Generation — Sinh tăng cường truy xuất) để đảm bảo trợ lý hỏi đáp của chúng ta có độ chính xác cao.

## HÔM NAY:

- Phần A: Chúng ta sẽ chia tài liệu thành các ĐOẠN
- Phần B: Chúng ta sẽ mã hóa các ĐOẠN thành VECTƠ và lưu vào Chroma
- Phần C: Chúng ta sẽ trực quan hóa các vectơ

### PHẦN A: Chia tài liệu thành các đoạn

In [ ]:
import os  # Nạp os để sử dụng trong notebook.
import glob  # Nạp glob để sử dụng trong notebook.
import tiktoken  # Nạp tiktoken để sử dụng trong notebook.
import numpy as np  # Nạp numpy as np để sử dụng trong notebook.
from dotenv import load_dotenv  # Nhập load_dotenv từ gói dotenv.
from langchain_openai import OpenAIEmbeddings  # Nhập OpenAIEmbeddings từ gói langchain_openai.
from langchain_chroma import Chroma  # Nhập Chroma từ gói langchain_chroma.
from langchain_huggingface import HuggingFaceEmbeddings  # Nhập HuggingFaceEmbeddings từ gói langchain_huggingface.
from langchain_community.document_loaders import DirectoryLoader, TextLoader  # Nhập DirectoryLoader, TextLoader từ gói langchain_community.document_loaders.
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Nhập RecursiveCharacterTextSplitter từ gói langchain_text_splitters.
from sklearn.manifold import TSNE  # Nhập TSNE từ gói sklearn.manifold.
import plotly.graph_objects as go  # Nạp plotly.graph_objects as go để sử dụng trong notebook.

In [ ]:
# Giá cả là một yếu tố quan trọng với công ty, vì vậy chúng ta sẽ sử dụng một mô hình có chi phí thấp

MODEL = "gpt-4.1-nano"  # Chọn tên mô hình ngôn ngữ sẽ được sử dụng.
db_name = "vector_db"  # Đặt tên thư mục cơ sở dữ liệu vectơ.
load_dotenv(override=True)  # Nạp lại các biến trong tệp `.env` vào môi trường chạy.
openai_api_key = os.getenv('OPENAI_API_KEY')  # Đọc khóa API OpenAI từ biến môi trường.
if openai_api_key:  # Kiểm tra khóa API OpenAI đã được cấu hình hay chưa.
    print(f"Khóa API OpenAI tồn tại và bắt đầu bằng {openai_api_key[:8]}")  # Thông báo đã tìm thấy khóa API mà không in toàn bộ khóa.
else:  # Xử lý trường hợp điều kiện phía trên không đúng.
    print("Chưa thiết lập khóa API OpenAI")  # In giá trị hoặc thông báo này ra phần output của ô.


In [ ]:
# Có bao nhiêu ký tự trong toàn bộ tài liệu?

knowledge_base_path = "knowledge-base/**/*.md"  # Đặt mẫu đường dẫn để tìm mọi tài liệu Markdown.
files = glob.glob(knowledge_base_path, recursive=True)  # Tìm tất cả tệp trong cơ sở tri thức theo mẫu glob.
print(f"Tìm thấy {len(files)} tệp trong cơ sở tri thức")  # In giá trị hoặc thông báo này ra phần output của ô.

entire_knowledge_base = ""  # Khởi tạo chuỗi để ghép toàn bộ nội dung cơ sở tri thức.

for file_path in files:  # Lặp qua từng phần tử của tập dữ liệu này.
    with open(file_path, 'r', encoding='utf-8') as f:  # Mở tệp hiện tại ở chế độ đọc văn bản UTF-8 và tự động đóng sau khi dùng.
        entire_knowledge_base += f.read()  # Nối nội dung hoặc dấu phân cách vào chuỗi cơ sở tri thức tổng hợp.
        entire_knowledge_base += "\n\n"  # Nối nội dung hoặc dấu phân cách vào chuỗi cơ sở tri thức tổng hợp.

print(f"Tổng số ký tự trong cơ sở tri thức: {len(entire_knowledge_base):,}")  # In giá trị hoặc thông báo này ra phần output của ô.

In [ ]:
# Có bao nhiêu token trong toàn bộ tài liệu?

encoding = tiktoken.encoding_for_model(MODEL)  # Lấy bộ mã hóa token phù hợp với mô hình đã chọn.
tokens = encoding.encode(entire_knowledge_base)  # Mã hóa toàn bộ cơ sở tri thức thành các token.
token_count = len(tokens)  # Đếm tổng số token đã mã hóa.
print(f"Tổng số token cho {MODEL}: {token_count:,}")  # In giá trị hoặc thông báo này ra phần output của ô.

In [ ]:
# Nạp toàn bộ nội dung trong cơ sở tri thức bằng các trình nạp của LangChain

folders = glob.glob("knowledge-base/*")  # Lấy danh sách các thư mục con trong cơ sở tri thức.

documents = []  # Khởi tạo danh sách chứa các tài liệu được nạp.
for folder in folders:  # Lặp qua từng thư mục loại tài liệu.
    doc_type = os.path.basename(folder)  # Lấy loại tài liệu từ tên thư mục cha.
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})  # Khởi tạo trình nạp các tệp Markdown trong thư mục hiện tại.
    folder_docs = loader.load()  # Nạp các tài liệu của thư mục hiện tại vào bộ nhớ.
    for doc in folder_docs:  # Lặp qua từng tài liệu vừa được nạp.
        doc.metadata["doc_type"] = doc_type  # Gắn loại tài liệu vào metadata để dùng khi truy xuất và trực quan hóa.
        documents.append(doc)  # Thêm tài liệu cùng loại, nguồn và nội dung vào danh sách.

print(f"Đã nạp {len(documents)} tài liệu")  # In giá trị hoặc thông báo này ra phần output của ô.

In [ ]:
documents[1]  # Hiển thị giá trị này trong output của notebook.

In [ ]:
# Chia thành các đoạn bằng RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)  # Cấu hình bộ chia văn bản với kích thước và độ chồng lấn đã chọn.
chunks = text_splitter.split_documents(documents)  # Tạo hoặc lưu danh sách các đoạn tài liệu.

print(f"Đã chia thành {len(chunks)} đoạn")  # In giá trị hoặc thông báo này ra phần output của ô.
print(f"Đoạn đầu tiên:\n\n{chunks[0]}")  # In giá trị hoặc thông báo này ra phần output của ô.

In [ ]:
chunks[100]  # Hiển thị giá trị này trong output của notebook.

### PHẦN B: Tạo vectơ và lưu vào Chroma

Ở Tuần 3, bạn đã thiết lập một tài khoản Hugging Face và nhận được `HF_TOKEN`.

Lúc này, bạn có thể thêm nó vào tệp `.env` rồi chạy `load_dotenv(override=True)`.

(Thực ra việc này có lẽ không bắt buộc.)

In [ ]:
# Chọn một mô hình embedding

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # Khởi tạo mô hình dùng để tạo embedding cho văn bản.
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):  # Kiểm tra cơ sở dữ liệu vectơ đã tồn tại hay chưa.
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()  # Xóa collection cũ để tránh trộn với dữ liệu của lần chạy trước.

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)  # Kết nối hoặc tạo kho vectơ Chroma.
print(f"Kho vectơ đã được tạo với {vectorstore._collection.count()} tài liệu")  # In giá trị hoặc thông báo này ra phần output của ô.

In [ ]:
# Hãy khảo sát các vectơ

collection = vectorstore._collection  # Lấy đối tượng collection để đọc hoặc ghi dữ liệu Chroma.
count = collection.count()  # Đếm số phần tử hiện có trong collection.

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]  # Lấy một embedding mẫu để kiểm tra số chiều.
dimensions = len(sample_embedding)  # Đếm số chiều của embedding mẫu.
print(f"Có {count:,} vectơ với {dimensions:,} chiều trong kho vectơ")  # In giá trị hoặc thông báo này ra phần output của ô.

### Phần C: Trực quan hóa!

In [ ]:
# Chuẩn bị

result = collection.get(include=['embeddings', 'documents', 'metadatas'])  # Lưu kết quả của bước xử lý hiện tại.
vectors = np.array(result['embeddings'])  # Chuyển danh sách embedding thành mảng NumPy.
documents = result['documents']  # Khởi tạo danh sách chứa các tài liệu được nạp.
metadatas = result['metadatas']  # Lấy metadata tương ứng với các tài liệu đã truy xuất.
doc_types = [metadata['doc_type'] for metadata in metadatas]  # Trích loại tài liệu từ metadata để phân nhóm dữ liệu.
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]  # Ánh xạ mỗi loại tài liệu sang một màu trên biểu đồ.

In [ ]:
# Con người chúng ta thấy việc trực quan hóa mọi thứ trong không gian 2D dễ hơn!
# Giảm số chiều của các vectơ xuống 2D bằng t-SNE
# (phép nhúng láng giềng ngẫu nhiên phân phối t)

tsne = TSNE(n_components=2, random_state=42)  # Khởi tạo t-SNE để giảm số chiều của vectơ.
reduced_vectors = tsne.fit_transform(vectors)  # Giảm các vectơ xuống số chiều cần trực quan hóa.

# Tạo biểu đồ phân tán 2D
fig = go.Figure(data=[go.Scatter(  # Tạo đối tượng biểu đồ Plotly.
    x=reduced_vectors[:, 0],  # Gán dữ liệu tọa độ cho trục x của biểu đồ.
    y=reduced_vectors[:, 1],  # Gán dữ liệu tọa độ cho trục y của biểu đồ.
    mode='markers',  # Chọn chế độ hiển thị bằng các điểm đánh dấu.
    marker=dict(size=5, color=colors, opacity=0.8),  # Cấu hình kích thước, màu và độ trong suốt của các điểm.
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],  # Tạo nội dung mô tả hiển thị khi rê chuột qua từng điểm.
    hoverinfo='text'  # Yêu cầu Plotly dùng phần văn bản làm thông tin khi rê chuột.
)])  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

fig.update_layout(title='Trực quan hóa kho vectơ Chroma trong không gian 2D',  # Cấu hình tiêu đề, trục, kích thước và lề của biểu đồ.
    scene=dict(xaxis_title='x',yaxis_title='y'),  # Đặt nhãn cho các trục trong vùng biểu đồ.
    width=800,  # Đặt chiều rộng của biểu đồ theo pixel.
    height=600,  # Đặt chiều cao của biểu đồ theo pixel.
    margin=dict(r=20, b=10, l=10, t=40)  # Đặt khoảng lề xung quanh biểu đồ.
)  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

fig.show()  # Hiển thị biểu đồ Plotly trong notebook.

In [ ]:
# Hãy thử với không gian 3D!

tsne = TSNE(n_components=3, random_state=42)  # Khởi tạo t-SNE để giảm số chiều của vectơ.
reduced_vectors = tsne.fit_transform(vectors)  # Giảm các vectơ xuống số chiều cần trực quan hóa.

# Tạo biểu đồ phân tán 3D
fig = go.Figure(data=[go.Scatter3d(  # Tạo đối tượng biểu đồ Plotly.
    x=reduced_vectors[:, 0],  # Gán dữ liệu tọa độ cho trục x của biểu đồ.
    y=reduced_vectors[:, 1],  # Gán dữ liệu tọa độ cho trục y của biểu đồ.
    z=reduced_vectors[:, 2],  # Gán dữ liệu tọa độ cho trục z của biểu đồ.
    mode='markers',  # Chọn chế độ hiển thị bằng các điểm đánh dấu.
    marker=dict(size=5, color=colors, opacity=0.8),  # Cấu hình kích thước, màu và độ trong suốt của các điểm.
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],  # Tạo nội dung mô tả hiển thị khi rê chuột qua từng điểm.
    hoverinfo='text'  # Yêu cầu Plotly dùng phần văn bản làm thông tin khi rê chuột.
)])  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

fig.update_layout(  # Cấu hình tiêu đề, trục, kích thước và lề của biểu đồ.
    title='Trực quan hóa kho vectơ Chroma trong không gian 3D',  # Đặt tiêu đề hiển thị cho biểu đồ.
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),  # Đặt nhãn cho các trục trong vùng biểu đồ.
    width=900,  # Đặt chiều rộng của biểu đồ theo pixel.
    height=700,  # Đặt chiều cao của biểu đồ theo pixel.
    margin=dict(r=10, b=10, l=10, t=40)  # Đặt khoảng lề xung quanh biểu đồ.
)  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

fig.show()  # Hiển thị biểu đồ Plotly trong notebook.